In [ ]:
app_fqdn = "mtg.test"
proxy_address = "127.0.0.1"
tenant_identifier = "sample"
owner_username = "owner"
owner_password = "E2e-Owner-Pass1"
owner_display_name = "Owner"
member_username = "member"
member_password = "E2e-Member-Pass1"
member_display_name = "Member"
room_title = "E2E Room"
default_result_path = None
close_on_fail = False
transition_timeout = 10000
connection_timeout = 60000

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# WebMeeting E2E Test - Meeting

- Test data to prepare: The account member_username and the room room_title, created by 02_Tenant_Management.ipynb

The browsers send the fake camera and microphone of Chromium.

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(
    close_on_fail=close_on_fail,
    last_path=default_result_path,
    ignore_https_errors=True,
    browser_args=[
        f"--host-resolver-rules=MAP {app_fqdn} {proxy_address}, MAP *.{app_fqdn} {proxy_address}",
        "--use-fake-ui-for-media-stream",
        "--use-fake-device-for-media-stream",
    ],
)

In [ ]:
# Keeps the RTCPeerConnection objects of the page so that the steps can read their state
track_peer_connections = """
(() => {
  const Original = window.RTCPeerConnection;
  window.__peerConnections = [];
  function Tracked(...args) {
    const pc = new Original(...args);
    window.__peerConnections.push(pc);
    return pc;
  }
  Tracked.prototype = Original.prototype;
  Object.setPrototypeOf(Tracked, Original);
  window.RTCPeerConnection = Tracked;
})();
"""

# True when a video element on the screen has presented a frame of a track received by
# a connected RTCPeerConnection. The page also has video elements of zero size.
remote_video_shown = """
() => {
  const received = new Set(window.__peerConnections
    .filter(pc => pc.connectionState === 'connected')
    .flatMap(pc => pc.getReceivers().map(r => r.track.id)));
  const videos = Array.from(document.querySelectorAll('video')).filter(v =>
    v.srcObject && v.getBoundingClientRect().width > 0 &&
    v.srcObject.getVideoTracks().some(t => received.has(t.id)));
  for (const v of videos) {
    if (v.__presentedFrames === undefined) {
      v.__presentedFrames = 0;
      const count = (now, metadata) => {
        v.__presentedFrames = metadata.presentedFrames;
        v.requestVideoFrameCallback(count);
      };
      v.requestVideoFrameCallback(count);
    }
  }
  return videos.some(v => v.__presentedFrames > 0);
}
"""

tenant_url = f"https://{tenant_identifier}.{app_fqdn}"
media_permissions = ["camera", "microphone"]

## Log in to the tenant as the owner.

In [ ]:
owner_page = None

async def _step(page):
    await page.context.add_init_script(track_peer_connections)
    await page.goto(f"{tenant_url}/login")
    await page.locator("input[name=username]").fill(owner_username)
    await page.locator("input[name=password]").fill(owner_password)
    await page.locator("button[type=submit]").click()

    await expect(page).to_have_url(f"{tenant_url}/home", timeout=transition_timeout)

await run_pw(_step, permissions=media_permissions)

## Open the room management, and click the join button of room_title.

The room opens in a new window with the button to enter the room.

In [ ]:
import re

room_id = None

async def _step(page):
    await page.get_by_role("link", name="ルーム管理").click()
    await expect(page).to_have_url(f"{tenant_url}/room", timeout=transition_timeout)

    row = page.locator("div.d-flex", has_text=room_title).filter(has=page.get_by_role("link", name="入室")).last
    popup_future = page.wait_for_event("popup")
    await row.get_by_role("link", name="入室").click()
    popup = await popup_future

    await expect(popup).to_have_url(re.compile(f"^{tenant_url}/r/[0-9]{{3}}-[0-9]{{3}}-[0-9]{{4}}$"), timeout=transition_timeout)
    await expect(popup.get_by_role("button", name="この設定で入室する")).to_be_visible(timeout=transition_timeout)

    global owner_page, room_id
    owner_page = popup
    room_id = popup.url.rsplit("/", 1)[1]
    return popup

await run_pw(_step)

## Enter the room as the owner.

In [ ]:
async def _step(page):
    await page.get_by_role("button", name="この設定で入室する").click()

    await expect(page.get_by_role("button", name="この設定で入室する")).to_be_hidden(timeout=transition_timeout)

await run_pw(_step)

## Log in to the tenant as the member in another browser, and enter the room ID on the home page.

The room opens in a new window with the button to enter the room.

In [ ]:
member_page = None

async def _step(page):
    await page.context.add_init_script(track_peer_connections)
    await page.goto(f"{tenant_url}/login")
    await page.locator("input[name=username]").fill(member_username)
    await page.locator("input[name=password]").fill(member_password)
    await page.locator("button[type=submit]").click()
    await expect(page).to_have_url(f"{tenant_url}/home", timeout=transition_timeout)

    await page.locator("#roomId").fill(room_id.replace("-", ""))
    popup_future = page.wait_for_event("popup")
    await page.locator("#enterButton").click()
    popup = await popup_future

    await expect(popup).to_have_url(f"{tenant_url}/r/{room_id}", timeout=transition_timeout)
    await expect(popup.get_by_role("button", name="この設定で入室する")).to_be_visible(timeout=transition_timeout)

    global member_page
    member_page = popup
    return popup

await run_pw(_step, new_context=True, permissions=media_permissions)

## Enter the room as the member, and verify that the member shows the video of the owner received over WebRTC.

In [ ]:
async def _step(page):
    await page.get_by_role("button", name="この設定で入室する").click()

    await page.wait_for_function(remote_video_shown, timeout=connection_timeout)
    await expect(page.locator("body")).to_contain_text(owner_display_name, timeout=transition_timeout)

await run_pw(_step)

## Verify that the owner shows the video of the member received over WebRTC.

In [ ]:
async def _step(page):
    await owner_page.wait_for_function(remote_video_shown, timeout=connection_timeout)
    await expect(owner_page.locator("body")).to_contain_text(member_display_name, timeout=transition_timeout)

    await owner_page.screenshot(path=f"{work_dir}/owner.png")

await run_pw(_step)

In [ ]:
from IPython.display import Image

Image(f"{work_dir}/owner.png")

Clean up

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}